In [19]:
!pip install jpegio

import os, torch, cv2, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import jpegio as jio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.ndimage import gaussian_filter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using Device: {DEVICE}")

 Using Device: cuda


In [20]:
import shutil
import os

# Working directory saaf karo
working_dir = '/kaggle/working'
for item in os.listdir(working_dir):
    item_path = os.path.join(working_dir, item)
    try:
        if os.path.isfile(item_path) or os.path.is_link(item_path):
            os.unlink(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
        print(f" Deleted: {item}")
    except Exception as e:
        print(f" Failed to delete {item_path}: {e}")

print(" Working directory Clean!")

 Deleted: best_model.pth
 Failed to delete /kaggle/working/.virtual_documents: module 'posixpath' has no attribute 'is_link'
 Working directory Clean!


In [21]:
import os, torch, cv2, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import jpegio as jio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.ndimage import gaussian_filter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DenseNet-121 Architecture: Growth rate k=32 [cite: 12, 219]
class DenseLayer(nn.Module):
    def __init__(self, in_ch, k):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, 4*k, 1, bias=False),
            nn.BatchNorm2d(4*k), nn.ReLU(inplace=True),
            nn.Conv2d(4*k, k, 3, padding=1, bias=False)
        )
    def forward(self, x): 
        return torch.cat([x, self.net(x)], 1)

class DenseNet121_JPEG(nn.Module):
    def __init__(self, num_classes=2, k=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), 
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        in_ch, config = 64, [6, 12, 24, 16] 
        self.blocks = nn.ModuleList()
        for i, nl in enumerate(config):
            block = nn.Sequential(*[DenseLayer(in_ch + j*k, k) for j in range(nl)])
            self.blocks.append(block)
            in_ch += nl * k
            if i < 3:
                trans = nn.Sequential(
                    nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
                    nn.Conv2d(in_ch, in_ch//2, 1, bias=False), 
                    nn.AvgPool2d(2, stride=2)
                )
                self.blocks.append(trans)
                in_ch //= 2
        self.final_bn = nn.BatchNorm2d(in_ch)
        self.classifier = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stem(x)
        for b in self.blocks: x = b(x)
        x = F.relu(self.final_bn(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.classifier(x)

In [22]:
def extract_features(path, patch_coords=None):
    try:
        jpg = jio.read(path)
        # 256x256 patch extraction [cite: 14, 280]
        if patch_coords:
            y, x = patch_coords
            dct = jpg.coef_arrays[0][y:y+256, x:x+256]
        else:
            dct = jpg.coef_arrays[0][:256, :256] 
        
        q = jpg.quant_tables[0].reshape(8,8) if jpg.quant_tables[0].ndim==1 else jpg.quant_tables[0]
        if dct.shape != (256, 256): return None

        b = 80 # Histogram range [-100, 100] [cite: 285, 329]
        bins = np.arange(-b, b+2) - 0.5
        H = np.zeros((64, 2*b+1), dtype=np.float32)
        Q = np.zeros((64, 2*b+1), dtype=np.float32)

        for idx, (u, v) in enumerate((u,v) for u in range(8) for v in range(8)):
            c = np.clip(dct[u::8, v::8].flatten(), -b, b)
            H[idx], _ = np.histogram(c, bins=bins)
            Q[idx] = float(q[u,v]) # Repeated q-factors [cite: 178]
            
        X = np.stack([H, Q], axis=-1) # Dimension 64 x 201 x 2 [cite: 185]
        X /= (X.max() + 1e-6)
        return X
    except: return None

class JPEGDataset(Dataset):
    def __init__(self, root_dir):
        self.file_list = []
        for label in ['single', 'double']:
            path = os.path.join(root_dir, label)
            if os.path.exists(path):
                for f in os.listdir(path):
                    self.file_list.append((os.path.join(path, f), 0 if label=='single' else 1))

    def __len__(self): return len(self.file_list)
    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        X = extract_features(path)
        if X is None: return torch.zeros((2, 64, 201)), label
        return torch.tensor(X).permute(2, 0, 1), label

In [27]:
import os
import torch
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import torch.nn as nn

# --- UPDATE THIS PATH ---
COLUMBIA_DATA_DIR = '/kaggle/input/datasets/sh3001/ucid-dataset/UCID_dataset' 

print("Loading local dataset...")
full_dataset = JPEGDataset(COLUMBIA_DATA_DIR)

# Safety check
assert len(full_dataset) > 0, f"No images found! Check your path: {COLUMBIA_DATA_DIR}"

# Split the dataset safely to avoid ValueError
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
# Force test_size to be the exact remaining remainder
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Load them straight into DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Setup Model, Optimizer, and Loss
model = DenseNet121_JPEG().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0

print(f" Training Start (20 Epochs) | Train: {len(train_dataset)}, Val: {len(val_dataset)}...")

for epoch in range(20):
    if epoch == 15: # LR Scheduler
        for pg in optimizer.param_groups: pg['lr'] = 0.0005

    # ==========================
    # TRAINING PHASE
    # ==========================
    model.train()
    train_loss = 0.0
    tp, tn, fp, fn = 0, 0, 0, 0 
    
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        
        # Calculate Training Metrics
        tp += ((preds == 1) & (labels == 1)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

    train_acc = (tp + tn) / (tp + tn + fp + fn + 1e-7)
    
    # ==========================
    # VALIDATION PHASE
    # ==========================
    model.eval()
    val_loss = 0.0
    v_tp, v_tn, v_fp, v_fn = 0, 0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            
            # Calculate Validation Metrics
            v_tp += ((preds == 1) & (labels == 1)).sum().item()
            v_tn += ((preds == 0) & (labels == 0)).sum().item()
            v_fp += ((preds == 1) & (labels == 0)).sum().item()
            v_fn += ((preds == 0) & (labels == 1)).sum().item()

    # Paper Formulas for Validation
    val_acc = (v_tp + v_tn) / (v_tp + v_tn + v_fp + v_fn + 1e-7)
    val_tpr = v_tp / (v_tp + v_fn + 1e-7)
    val_tnr = v_tn / (v_tn + v_fp + 1e-7)
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f" Epoch {epoch+1}")
    print(f"   Train | Loss: {avg_train_loss:.4f} | Acc: {train_acc*100:.2f}%")
    print(f"   Val   | Loss: {avg_val_loss:.4f} | Acc: {val_acc*100:.2f}% | TPR: {val_tpr*100:.2f}% | TNR: {val_tnr*100:.2f}%")
    
    # Save the model only if validation accuracy improves
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        print(f"New best model saved! (Val Acc: {best_val_acc*100:.2f}%)")

Loading local dataset...
 Training Start (20 Epochs) | Train: 3746, Val: 802...


Epoch 1 [Val]: 100%|██████████| 13/13 [00:13<00:00,  1.00s/it]


 Epoch 1
   Train | Loss: 0.5124 | Acc: 71.89%
   Val   | Loss: 0.6514 | Acc: 64.84% | TPR: 33.91% | TNR: 96.23%
New best model saved! (Val Acc: 64.84%)


Epoch 2 [Val]: 100%|██████████| 13/13 [00:06<00:00,  1.98it/s]


 Epoch 2
   Train | Loss: 0.4480 | Acc: 75.57%
   Val   | Loss: 0.5791 | Acc: 66.71% | TPR: 40.10% | TNR: 93.72%
New best model saved! (Val Acc: 66.71%)


...

 Epoch 18
   Train | Loss: 0.3703 | Acc: 81.58%
   Val   | Loss: 0.6228 | Acc: 62.72% | TPR: 97.03% | TNR: 27.89%


Epoch 19 [Val]: 100%|██████████| 13/13 [00:06<00:00,  2.03it/s]


 Epoch 19
   Train | Loss: 0.3567 | Acc: 82.46%
   Val   | Loss: 0.3576 | Acc: 80.55% | TPR: 77.97% | TNR: 83.17%


Epoch 20 [Val]: 100%|██████████| 13/13 [00:05<00:00,  2.23it/s]

 Epoch 20
   Train | Loss: 0.3435 | Acc: 82.97%
   Val   | Loss: 0.3878 | Acc: 78.18% | TPR: 56.93% | TNR: 99.75%


In [28]:
import torch
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

# 1. Load the best weights saved during the training phase
model_path = '/kaggle/working/best_model.pth'
try:
    model.load_state_dict(torch.load(model_path))
    print(f"Successfully loaded best model weights from {model_path}")
except FileNotFoundError:
    print(f"Error: Could not find {model_path}. Make sure the training cell finished saving it.")

# 2. Set model to evaluation mode (turns off dropout, batchnorm updates, etc.)
model.eval()

# 3. Tracking variables for paper metrics
y_true = []
y_scores = [] # Required for ROC AUC curve
v_tp, v_tn, v_fp, v_fn = 0, 0, 0, 0

print(f"\nInitiating testing on the unseen Columbia dataset (Test Set Size: {len(test_dataset)} patches)...")

# 4. Run Inference without calculating gradients (saves memory & speeds up)
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        outputs = model(inputs)
        
        # Use softmax to get the probability score for class 1 (double compressed)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)
        
        # Store for Scikit-Learn metrics
        y_true.extend(labels.cpu().numpy())
        y_scores.extend(probs.cpu().numpy())
        
        # Calculate Confusion Matrix values explicitly
        v_tp += ((preds == 1) & (labels == 1)).sum().item()
        v_tn += ((preds == 0) & (labels == 0)).sum().item()
        v_fp += ((preds == 1) & (labels == 0)).sum().item()
        v_fn += ((preds == 0) & (labels == 1)).sum().item()

# 5. Calculate the exact metrics reported in the paper
test_acc = (v_tp + v_tn) / (v_tp + v_tn + v_fp + v_fn + 1e-7)
test_tpr = v_tp / (v_tp + v_fn + 1e-7)  # TPR is mathematically identical to Recall
test_tnr = v_tn / (v_tn + v_fp + 1e-7)

precision = v_tp / (v_tp + v_fp + 1e-7)
recall = test_tpr
f_measure = 2 * (precision * recall) / (precision + recall + 1e-7)

# Calculate Area Under the Receiver Operating Characteristic Curve (ROC AUC)
roc_auc = roc_auc_score(y_true, y_scores)

# 6. Display Results formatted like the paper's tables
print("\n" + "="*50)
print(" FINAL TESTING METRICS (UNSEEN MATRICES)")
print("="*50)
print(f" Accuracy:       {test_acc*100:.2f}%")
print(f" TPR (Recall):   {test_tpr*100:.2f}%  <- % of double compressed correctly found")
print(f" TNR:            {test_tnr*100:.2f}%  <- % of single compressed correctly found")
print("-" * 50)
print(f" Precision:      {precision:.4f}")
print(f" F-Measure:      {f_measure:.4f}")
print(f" ROC AUC:        {roc_auc:.4f}")
print("="*50)

Successfully loaded best model weights from /kaggle/working/best_model.pth

Initiating testing on the unseen Columbia dataset (Test Set Size: 804 patches)...


Testing: 100%|██████████| 13/13 [00:13<00:00,  1.01s/it]


 FINAL TESTING METRICS (UNSEEN MATRICES)
 Accuracy:       78.86%
 TPR (Recall):   72.38%  <- % of double compressed correctly found
 TNR:            84.99%  <- % of single compressed correctly found
--------------------------------------------------
 Precision:      0.8203
 F-Measure:      0.7690
 ROC AUC:        0.8842
